Installations

In [ ]:
!pip install -U langchain langchain-groq langchain-community \
    langchain-text-splitters langchain-huggingface \
    sentence-transformers faiss-cpu pypdf pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existi

Imports

In [ ]:
import os
import tempfile                       # for cv uplaoding
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_groq import ChatGroq

from getpass import getpass
from collections import Counter

/tmp/ipykernel_5626/892268001.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDFs path for RAG

In [ ]:
static_pdf_paths = [
    "data/interview_tips.pdf",
    "data/cv_writing_tips.pdf"
]

Read PDFs

In [ ]:
pdf_docs = []

for path in static_pdf_paths:
    loader = PyPDFLoader(path)
    pdf_docs.extend(loader.load())

print(f"Loaded {len(pdf_docs)} pages from PDFs")

Loaded 24 pages from PDFs


CV Uplaoding

In [ ]:
from google.colab import files

uploaded = files.upload()

cv_filename = list(uploaded.keys())[0]

print(f"Uploaded CV: {cv_filename}")

Saving my_cv.pdf to my_cv.pdf
Uploaded CV: my_cv.pdf


In [ ]:
cv_loader = PyPDFLoader(cv_filename)
cv_docs = cv_loader.load()

for doc in cv_docs:
    doc.metadata["source_type"] = "cv"

print(f"Loaded {len(cv_docs)} CV pages")

Loaded 2 CV pages


Read Jobs file

In [ ]:
df = pd.read_csv("data/jobs.csv")

print(df.head())
print(f"\nLoaded {len(df)} job entries")

                           Title  \
0                   Data Analyst   
1        Machine Learning Intern   
2                 Data Scientist   
3                    AI Engineer   
4  Business Intelligence Analyst   

                                           Skills Experience Location  
0                    Python, SQL, Excel, Power BI  0-2 years    Cairo  
1  Python, Scikit-learn, Pandas, Machine Learning  0-1 years    Cairo  
2       Python, SQL, Machine Learning, Statistics  1-3 years    Cairo  
3      Python, PyTorch, TensorFlow, Deep Learning  0-2 years    Cairo  
4        Power BI, SQL, Excel, Data Visualization  0-2 years    Cairo  

Loaded 5 job entries


Convert every job to text (Because RAG needs text not rows)

In [ ]:
job_texts = []

for _, row in df.iterrows():
    text = (
        f"Job Title: {row['Title']}. "
        f"Required Skills: {row['Skills']}. "
        f"Experience: {row['Experience']}. "
        f"Location: {row['Location']}."
    )
    job_texts.append(text)

print(f"Loaded {len(job_texts)} job entries")

Loaded 5 job entries


Chunking PDFs using recursive chunking for preserving the meaning

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

static_chunks = splitter.split_documents(pdf_docs)

cv_chunks = splitter.split_documents(cv_docs)

print(f"Created {len(static_chunks)} static PDF chunks")
print(f"Created {len(cv_chunks)} CV chunks")

Created 138 static PDF chunks
Created 15 CV chunks


Set source type (Metadata)

In [ ]:
for doc in static_chunks:

    source = doc.metadata.get("source", "").lower()

    if "cv_writing_tips" in source:
        doc.metadata["source_type"] = "cv_guide"

    elif "interview_tips" in source:
        doc.metadata["source_type"] = "interview_guide"


print("Source types assigned successfully")

type_counts = Counter(
    doc.metadata.get("source_type", "UNSET")
    for doc in static_chunks + cv_chunks
)

print("Chunk counts:", dict(type_counts))

Source types assigned successfully
Chunk counts: {'interview_guide': 66, 'cv_guide': 72, 'cv': 15}


Converting job data into Documents to combine with PDF documents before creating the vector store.

In [ ]:
job_chunks = []

for i, text in enumerate(job_texts):

    job_chunks.append(
        Document(
            page_content=text,
            metadata={
                "source_type": "jobs",
                "job_number": i + 1
            }
        )
    )

print(f"Created {len(job_chunks)} job documents")

Created 5 job documents


Create Embeddings using all-MiniLM-L6-v2 model

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model loaded


Build FAISS Vector Store for embedding storage

In [ ]:
cv_vectorstore = FAISS.from_documents(
    cv_chunks,
    embeddings
)

jobs_vectorstore = FAISS.from_documents(
    job_chunks,
    embeddings
)

cv_guide_vectorstore = FAISS.from_documents(
    [
        doc for doc in static_chunks
        if doc.metadata.get("source_type") == "cv_guide"
    ],
    embeddings
)

interview_vectorstore = FAISS.from_documents(
    [
        doc for doc in static_chunks
        if doc.metadata.get("source_type") == "interview_guide"
    ],
    embeddings
)

print("All vector stores created successfully")

All vector stores created successfully


Create Retrivers

In [ ]:
cv_retriever = cv_vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

jobs_retriever = jobs_vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

cv_guide_retriever = cv_guide_vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

interview_retriever = interview_vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("All retrievers ready")

All retrievers ready


Calling Groq LLM

In [ ]:
groq_api_key = getpass("Enter your Groq API key: ")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=groq_api_key,
    temperature=0
)

print("LLM initialized successfully")

Enter your Groq API key: ··········
LLM initialized successfully


Memory

In [ ]:
chat_history = []

def format_history(history, max_turns=5):

    if not history:
        return "No previous conversation."

    recent = history[-max_turns:]

    return "\n".join(
        f"User: {q}\nAssistant: {a}"
        for q, a in recent
    )

print("Memory initialized")

Memory initialized


Prompt

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful Career Assistant.

Your job is to answer the user's question using ONLY the provided
Current Context and Conversation History.

STRICT RULES:

1. Do NOT use outside knowledge.
2. Do NOT invent, guess, assume, or complete missing information.
3. Every factual statement in your answer must be supported by the Current Context.
4. If the requested information is not present in the Current Context,
   clearly say that the information is not available.
5. Never use information from one source as if it came from another source.
6. When the question is about the user's CV, use only information from
   the user's CV.
7. Do not use example resumes from the CV writing guide as the user's CV.
8. When the user refers to "first job", "second job", "third job", etc.,
   use the numbered job information provided in the Current Context.
9. When the user uses words such as "it", "that job", or "this job",
   use Conversation History only to understand what they are referring to.
10. If the Current Context conflicts with Conversation History,
    always prefer the Current Context.
11. When comparing the CV with jobs, clearly distinguish:
    - Matching skills
    - Missing skills
    - Information that is not provided
12. Do not provide salary, experience, skills, qualifications, or other
    details unless they are explicitly present in the Current Context.

Conversation History:
{chat_history}

Current Context:
{context}
"""
    ),
    ("human", "{input}")
])

Retrieve Documents

In [ ]:
def retrieve_documents(question):

    question_lower = question.lower()

    # Handle numbered job questions
    job_numbers = {
        "first": 0,
        "second": 1,
        "third": 2,
        "fourth": 3,
        "fifth": 4
    }

    for word, index in job_numbers.items():

        if f"{word} job" in question_lower:

            job_docs = []

            for i, (_, row) in enumerate(df.iterrows()):

                job_docs.append(
                    Document(
                        page_content=(
                            f"Job {i + 1}: "
                            f"Job Title: {row['Title']}. "
                            f"Required Skills: {row['Skills']}. "
                            f"Experience: {row['Experience']}. "
                            f"Location: {row['Location']}."
                        ),
                        metadata={
                            "source_type": "jobs",
                            "job_number": i + 1
                        }
                    )
                )

            return job_docs

    # Questions that require BOTH CV and Jobs
    if (
        ("job" in question_lower or "jobs" in question_lower)
        and ("my cv" in question_lower or "my resume" in question_lower)
    ):
        cv_docs = cv_retriever.invoke(question)
        job_docs = jobs_retriever.invoke(question)
        return cv_docs + job_docs

    # CV-related questions
    cv_keywords = [
        "my cv",
        "my resume",
        "cv owner",
        "resume owner",
        "name of cv owner",
        "name of resume owner",
        "my name",
        "my email",
        "my phone",
        "my education",
        "my skills",
        "my experience"
    ]

    if any(keyword in question_lower for keyword in cv_keywords):
        return cv_retriever.invoke(question)

    # Job questions
    elif "job" in question_lower or "jobs" in question_lower:
        return jobs_retriever.invoke(question)

    # Interview questions
    elif "interview" in question_lower:
        return interview_retriever.invoke(question)

    # CV tips
    elif "cv tips" in question_lower or "resume tips" in question_lower:
        return cv_guide_retriever.invoke(question)

    # General questions
    else:
        return retriever.invoke(question)

General Vector Store

In [ ]:
all_chunks = (
    static_chunks
    + cv_chunks
    + job_chunks
)

general_vectorstore = FAISS.from_documents(
    all_chunks,
    embeddings
)

retriever = general_vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("General retriever ready")

General retriever ready


Connect Components with LangChain

In [ ]:
def format_docs(docs):

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {
        "context": RunnableLambda(
            lambda q: format_docs(
                retrieve_documents(q)
            )
        ),

        "chat_history": RunnableLambda(
            lambda q: format_history(chat_history)
        ),

        "input": RunnablePassthrough()
    }

    | prompt
    | llm
)

print("RAG chain created successfully")

RAG chain created successfully


Ask Function

In [ ]:
def ask(question):

    response = rag_chain.invoke(question)

    chat_history.append(
        (question, response.content)
    )

    return response.content

Testing

In [ ]:
chat_history = []

In [ ]:
print("Q1:", "What skills are mentioned in my CV?")
print("A1:", ask("What skills are mentioned in my CV?"))

Q1: What skills are mentioned in my CV?
A1: The skills listed in your CV are:

- Python  
- NumPy  
- Microsoft Power BI  
- Hyperparameter Tuning  
- Deep Learning  
- RAG‑Systems  
- Model Evaluation  
- OpenCV  
- Problem Solving  
- SQL  
- Pandas  
- Microsoft Excel  
- NLP  
- Agentic AI  
- LLMs  
- TensorFlow  
- Streamlit  
- Critical Thinking  
- C++  
- Data Visualization  
- Machine Learning  
- Computer Vision  
- Fine‑Tuning  
- LangChain  
- FastAPI  
- Teamwork  
- Work under pressure


In [ ]:
print("Q2:", "Which job from the list fits me best based on my CV?")
print("A2:", ask("Which job from the list fits me best based on my CV?"))

Q2: Which job from the list fits me best based on my CV?
A2: **Job(s) that match all of the required skills listed in your CV**

| Job Title | Required Skills (from the list) | Matching Skills in your CV | Missing Skills | Information Not Provided |
|-----------|--------------------------------|----------------------------|----------------|--------------------------|
| **Data Analyst** | Python, SQL, Excel, Power BI | Python, SQL, Microsoft Excel, Microsoft Power BI | – none – | – |
| **Business Intelligence Analyst** | Power BI, SQL, Excel, Data Visualization | Microsoft Power BI, SQL, Microsoft Excel, Data Visualization | – none – | – |

**Why these jobs fit best**

- **All required technical skills are present in your CV** for both the Data Analyst and Business Intelligence Analyst positions.  
- No required skill is missing for either of these roles.  

**Other jobs and the gaps**

| Job Title | Required Skills | Matching Skills | Missing Skills |
|-----------|----------------|----

In [ ]:
print("Q3:", "And what about the second job on the list?")
print("A3:", ask("And what about the second job on the list?"))

Q3: And what about the second job on the list?
A3: **Job 2 – Machine Learning Intern**

| Category | Details |
|----------|---------|
| **Required Skills** (from the job posting) | Python, Scikit‑learn, Pandas, Machine Learning |
| **Matching Skills in your CV** | Python, Pandas, Machine Learning |
| **Missing Skills** | Scikit‑learn (not listed in your CV) |
| **Information Not Provided in your CV** | • Your years of experience (the role requires 0‑1 years) <br>• Confirmation that you are based in Cairo (the role’s location) |

**Summary**

- You already have three of the four required technical skills for the Machine Learning Intern position.  
- The only gap is Scikit‑learn, which is not mentioned in your CV.  
- If you have experience with Scikit‑learn that is not reflected in the current CV, adding it would make you a complete match for this role.


In [ ]:
print("Q4:", "Does it require SQL?")
print("A4:", ask("Does it require SQL?"))

Q4: Does it require SQL?
A4: No. The **Machine Learning Intern** (the second job on the list) lists the required technical skills as **Python, Scikit‑learn, Pandas, and Machine Learning**. SQL is not mentioned among its required skills.


In [ ]:
print("Q5:", "What are some important CV writing tips?")
print("A5:", ask("What are some important CV writing tips?"))

Q5: What are some important CV writing tips?
A5: Here are the key CV‑writing tips that are mentioned in the provided guide:

| Tip | What the guide says |
|-----|----------------------|
| **Start with an action verb** | Use a strong verb to show you did something. |
| **Provide context** | Add quantitative or qualitative details that explain the situation for your action. |
| **Show the end result** | Explain the value or impact of your contribution. |
| **Verb tense** | Use **present tense** for current roles and **past tense** for former roles. |
| **Margins** | Set margins between **0.75 in** and **1 in** (ideal) and never less than **0.5 in**. |
| **Font style & size** | Keep the font style consistent and use a size between **10 pt and 12 pt**. |
| **Contact information** | Omit the mailing address; include only your phone number and email to save space. |
| **Order of content** | List items in **reverse chronological order** within each section. |
| **Readability** | Make the resu

In [ ]:
print("Q6:", "What are some important interview tips?")
print("A6:", ask("What are some important interview tips?"))

Q6: What are some important interview tips?
A6: Here are the interview tips that are listed in the provided interview guide:

| Tip | What the guide recommends |
|-----|----------------------------|
| **Set aside time to prepare** | Allocate dedicated time for each interview so you can get ready thoroughly. |
| **Practice answers to common questions** | Rehearse responses to typical interview questions to feel more confident and articulate. |
| **Understand the interview type and relevant questions** | Know the specific interview format and the kinds of questions that are common for the industry or job function you’re targeting. |
| **Develop 3‑5 go‑to stories** | Prepare a few concise stories that showcase both general career‑readiness competencies and the specific skills/traits required for the position you’re interviewing for. |

These points capture the key interview preparation advice from the current context.


In [ ]:
print("Q7:", "What is the salary of the AI Engineer job?")
print("A7:", ask("What is the salary of the AI Engineer job?"))

Q7: What is the salary of the AI Engineer job?
A7: The salary for the **AI Engineer** position is not provided in the current context.


In [ ]:
print("Q8:", "What is the name of CV owner?")
print("A8:", ask("What is the name of CV owner?"))

Q8: What is the name of CV owner?
A8: The CV owner’s name is **Karas Sherif Mina Gerges**.


In [ ]:
print("Q9:", "What is the phone number of the CV owner?")
print("A9:", ask("What is the phone number of the CV owner?"))

Q9: What is the phone number of the CV owner?
A9: The phone number of the CV owner is not provided in the current context.
